# 04 Train WLASL300 BiGRU + Temporal Attention Model

## Purpose

This notebook trains the selected WLASL100 architecture on WLASL300.

## Why this model

From WLASL100 experiments, **BiGRU + Temporal Attention** outperformed the baseline BiLSTM and the Small Transformer Encoder. Therefore, this notebook scales the winning architecture to 300 ASL signs.

## Model strategy

The model uses:

- MediaPipe keypoints
- Velocity/motion features
- Global train-set normalisation
- Balanced class sampling
- BiGRU sequence modelling
- Temporal attention pooling
- AdamW optimiser
- Learning-rate scheduling
- Gradient clipping
- Early stopping
- Top-1, Top-3, Top-5 accuracy
- Macro F1-score

In [ ]:
from pathlib import Path
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)

## 1. Set paths and training settings

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path("E:/Be_My_Ear")

DATASET_NAME = "WLASL300"
PREFIX = "wlasl300"

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
CLEAN_INDEX_FILE = BASE_DIR / f"{PREFIX}_clean_keypoint_index.csv"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL" / DATASET_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / f"bigru_attention_{PREFIX}.pt"
HISTORY_PATH = MODEL_DIR / f"bigru_attention_{PREFIX}_history.csv"
NORM_STATS_PATH = MODEL_DIR / f"{PREFIX}_train_norm_stats.npz"

BATCH_SIZE = 32
USE_VELOCITY = True
INPUT_SIZE = 516
SEQUENCE_LENGTH = 60
EPOCHS = 60
EARLY_STOPPING_PATIENCE = 15

print("Dataset:", DATASET_NAME)
print("Clean index exists:", CLEAN_INDEX_FILE.exists())
print("Model will save to:", MODEL_PATH)

## 2. Load clean WLASL300 dataset

In [ ]:
df = pd.read_csv(CLEAN_INDEX_FILE)

print("Clean samples:", len(df))
print("Classes:", df["label_id"].nunique())
print("Example keypoint path:", df.iloc[0]["keypoint_path"])

df.head()

## 3. Create train / validation / test split

In [ ]:
train_records = []
val_records = []
test_records = []

for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)

    n = len(group)
    n_test = max(1, int(round(n * 0.15)))
    n_val = max(1, int(round(n * 0.15)))

    test_records.append(group.iloc[:n_test])
    val_records.append(group.iloc[n_test:n_test + n_val])
    train_records.append(group.iloc[n_test + n_val:])

train_df = pd.concat(train_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Train classes:", train_df["label_id"].nunique())
print("Validation classes:", val_df["label_id"].nunique())
print("Test classes:", test_df["label_id"].nunique())

## 4. Compute global train-set normalisation

In [ ]:
def compute_train_normalisation_stats(train_dataframe):
    total_sum = None
    total_sq_sum = None
    total_count = 0

    for path in tqdm(train_dataframe["keypoint_path"], desc="Computing train mean/std"):
        arr = np.load(path).astype(np.float32)

        if total_sum is None:
            total_sum = arr.sum(axis=0)
            total_sq_sum = (arr ** 2).sum(axis=0)
        else:
            total_sum += arr.sum(axis=0)
            total_sq_sum += (arr ** 2).sum(axis=0)

        total_count += arr.shape[0]

    mean = total_sum / total_count
    variance = (total_sq_sum / total_count) - (mean ** 2)
    variance = np.maximum(variance, 1e-6)
    std = np.sqrt(variance)

    return mean.astype(np.float32), std.astype(np.float32)

train_mean, train_std = compute_train_normalisation_stats(train_df)
np.savez(NORM_STATS_PATH, mean=train_mean, std=train_std)

print("Saved normalisation stats to:", NORM_STATS_PATH)
print("Mean shape:", train_mean.shape)
print("Std shape:", train_std.shape)

## 5. Create PyTorch dataset and balanced data loaders

In [ ]:
class SignKeypointDataset(Dataset):
    def __init__(self, dataframe, mean, std, use_velocity=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mean = mean.reshape(1, -1).astype(np.float32)
        self.std = std.reshape(1, -1).astype(np.float32)
        self.use_velocity = use_velocity

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        keypoints = np.load(row["keypoint_path"]).astype(np.float32)
        keypoints = (keypoints - self.mean) / (self.std + 1e-6)

        if self.use_velocity:
            velocity = np.zeros_like(keypoints, dtype=np.float32)
            velocity[1:] = keypoints[1:] - keypoints[:-1]
            features = np.concatenate([keypoints, velocity], axis=1)
        else:
            features = keypoints

        label = int(row["label_id"])

        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

train_dataset = SignKeypointDataset(train_df, train_mean, train_std, USE_VELOCITY)
val_dataset = SignKeypointDataset(val_df, train_mean, train_std, USE_VELOCITY)
test_dataset = SignKeypointDataset(test_df, train_mean, train_std, USE_VELOCITY)

class_counts = train_df["label_id"].value_counts().to_dict()
sample_weights = train_df["label_id"].map(lambda label: 1.0 / class_counts[label]).values
sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))

print("Input batch shape:", x_batch.shape)
print("Label batch shape:", y_batch.shape)

## 6. Define BiGRU + Temporal Attention model

In [ ]:
class BiGRUAttentionModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.4):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)

        attention_scores = self.attention(gru_out).squeeze(-1)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)

        context = torch.sum(gru_out * attention_weights, dim=1)
        logits = self.classifier(context)

        return logits

## 7. Initialise model, optimiser, and scheduler

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = df["label_id"].nunique()

model = BiGRUAttentionModel(
    input_size=INPUT_SIZE,
    hidden_size=256,
    num_classes=NUM_CLASSES,
    num_layers=2,
    dropout=0.4
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=5
)

print("Using device:", device)
print("Number of classes:", NUM_CLASSES)
print(model)

## 8. Training and evaluation helper functions

In [ ]:
def top_k_accuracy(outputs, labels, k=5):
    _, top_k_preds = outputs.topk(k, dim=1)
    correct = top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds))
    return correct.any(dim=1).float().mean().item()

def run_epoch(model, loader, optimizer=None, phase="Train", epoch=1, total_epochs=1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0
    total_top1 = 0
    total_top3 = 0
    total_top5 = 0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [{phase}]", leave=False)

    with torch.set_grad_enabled(is_train):
        for step, (x, y) in enumerate(progress_bar, start=1):
            x = x.to(device)
            y = y.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(x)
            loss = criterion(outputs, y)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds = torch.argmax(outputs, dim=1)

            batch_top1 = (preds == y).float().mean().item()
            batch_top3 = top_k_accuracy(outputs, y, k=3)
            batch_top5 = top_k_accuracy(outputs, y, k=5)

            total_loss += loss.item()
            total_top1 += batch_top1
            total_top3 += batch_top3
            total_top5 += batch_top5

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

            progress_bar.set_postfix({
                "step": f"{step}/{len(loader)}",
                "loss": f"{loss.item():.4f}",
                "top1": f"{batch_top1:.4f}",
                "top5": f"{batch_top5:.4f}"
            })

    avg_loss = total_loss / len(loader)
    avg_top1 = total_top1 / len(loader)
    avg_top3 = total_top3 / len(loader)
    avg_top5 = total_top5 / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, avg_top1, avg_top3, avg_top5, macro_f1

## 9. Train WLASL300 model

In [ ]:
history = {
    "train_loss": [],
    "train_top1": [],
    "train_top3": [],
    "train_top5": [],
    "train_f1": [],
    "val_loss": [],
    "val_top1": [],
    "val_top3": [],
    "val_top5": [],
    "val_f1": [],
    "lr": []
}

best_val_f1 = 0.0
best_val_top5 = 0.0
epochs_without_improvement = 0

print("=" * 80)
print("Be My Ear - WLASL300 BiGRU + Temporal Attention Training")
print("=" * 80)
print(f"Device: {device}")
print(f"Input shape: (60, {INPUT_SIZE})")
print(f"Classes: {NUM_CLASSES}")
print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Model save path: {MODEL_PATH}")
print("=" * 80)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 80)

    train_loss, train_top1, train_top3, train_top5, train_f1 = run_epoch(
        model, train_loader, optimizer=optimizer, phase="Training", epoch=epoch, total_epochs=EPOCHS
    )

    val_loss, val_top1, val_top3, val_top5, val_f1 = run_epoch(
        model, val_loader, optimizer=None, phase="Validation", epoch=epoch, total_epochs=EPOCHS
    )

    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_top1"].append(train_top1)
    history["train_top3"].append(train_top3)
    history["train_top5"].append(train_top5)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_top1"].append(val_top1)
    history["val_top3"].append(val_top3)
    history["val_top5"].append(val_top5)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    improved = val_f1 > best_val_f1

    if improved:
        best_val_f1 = val_f1
        best_val_top5 = val_top5
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_f1": best_val_f1,
            "best_val_top5": best_val_top5,
            "num_classes": NUM_CLASSES,
            "input_size": INPUT_SIZE,
            "sequence_length": SEQUENCE_LENGTH,
            "use_velocity": USE_VELOCITY,
            "architecture": "BiGRUAttentionModel"
        }, MODEL_PATH)

        save_status = "Saved new best model"
    else:
        epochs_without_improvement += 1
        save_status = "No improvement"

    print(f"Train | Loss: {train_loss:.4f} | Top-1: {train_top1:.4f} | Top-3: {train_top3:.4f} | Top-5: {train_top5:.4f} | F1: {train_f1:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Top-1: {val_top1:.4f} | Top-3: {val_top3:.4f} | Top-5: {val_top5:.4f} | F1: {val_f1:.4f}")
    print(f"Learning rate: {current_lr:.8f}")
    print(f"Status: {save_status}")
    print(f"Best Val F1 so far: {best_val_f1:.4f}")
    print(f"Best Val Top-5 so far: {best_val_top5:.4f}")
    print(f"Epochs without improvement: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break

training_minutes = (time.time() - start_time) / 60

print("\n" + "=" * 80)
print("Training completed")
print("=" * 80)
print(f"Total training time: {training_minutes:.2f} minutes")
print(f"Best validation F1: {best_val_f1:.4f}")
print(f"Best validation Top-5: {best_val_top5:.4f}")
print(f"Best model saved to: {MODEL_PATH}")
print("=" * 80)

## 10. Save training history and evaluate test set

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print("Saved history to:", HISTORY_PATH)
history_df.head()

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_top1, test_top3, test_top5, test_f1 = run_epoch(
    model, test_loader, optimizer=None, phase="Test", epoch=1, total_epochs=1
)

print("=" * 80)
print("WLASL300 Test Evaluation")
print("=" * 80)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")
print("=" * 80)

In [ ]:
RESULT_FILE = MODEL_DIR / f"bigru_attention_{PREFIX}_result_summary.csv"

result_df = pd.DataFrame([{
    "dataset": DATASET_NAME,
    "model": "BiGRU + Temporal Attention",
    "clean_samples": len(df),
    "classes": NUM_CLASSES,
    "input_shape": f"(60, {INPUT_SIZE})",
    "velocity_features": USE_VELOCITY,
    "best_val_f1": checkpoint["best_val_f1"],
    "best_val_top5": checkpoint["best_val_top5"],
    "checkpoint_epoch": checkpoint["epoch"],
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_f1,
    "model_path": str(MODEL_PATH),
    "history_path": str(HISTORY_PATH)
}])

result_df.to_csv(RESULT_FILE, index=False)

print("Saved result summary to:", RESULT_FILE)
result_df

## Final summary

After this notebook, continue with:

```text
05_evaluate_wlasl300_bigru_attention.ipynb
```